In [3]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import os
import subprocess

# Common mount points at GIUB
possible_paths = [
    "/mnt/hydroshare/data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    "/media/hydroshare/data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    "/run/user/1000/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    "~/hydroshare/data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    # Additional GVFS patterns (auto-mounted network drives)
    f"/run/user/{os.getuid()}/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
]

# Also check for any GVFS mount dynamically
try:
    gvfs_path = Path(f"/run/user/{os.getuid()}/gvfs")
    if gvfs_path.exists():
        for mount in gvfs_path.iterdir():
            if 'hydroshare' in mount.name.lower():
                possible_file = mount / "Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc"
                if possible_file.exists():
                    possible_paths.insert(0, str(possible_file))
except Exception:
    pass

print("🔍 Searching for HAR file in common locations...")

# Check which path exists
har_file = None
for path in possible_paths:
    expanded_path = os.path.expanduser(path)
    if os.path.exists(expanded_path):
        har_file = expanded_path
        print(f"✅ Found file at: {har_file}")
        break

if har_file is None:
    print("❌ File not found at any common mount point!")
    print("\n🔧 Debugging information:")
    print("\nChecking mounted filesystems...")
    result = subprocess.run(['mount'], capture_output=True, text=True)
    smb_mounts = [line for line in result.stdout.split('\n') if 'hydroshare' in line.lower() or 'gvfs' in line.lower()]
    
    if smb_mounts:
        print("\n🔍 Found potential mounts:")
        for mount in smb_mounts:
            print(f"   {mount}")
    
    # Check GVFS mounts
    gvfs_base = Path(f"/run/user/{os.getuid()}/gvfs")
    if gvfs_base.exists():
        print(f"\n📁 GVFS mounts in {gvfs_base}:")
        for item in gvfs_base.iterdir():
            print(f"   {item.name}")
    
    print("\n💡 To access the file:")
    print("   1. Open your file manager (Files/Nautilus)")
    print("   2. Click on 'Other Locations' in sidebar")
    print("   3. Connect to: smb://hydroshare.giub.unibe.ch/data")
    print("   4. Browse to: Meteorology/HMA/")
    print("   5. Re-run this script - it should auto-detect the mount")
    
else:
    # File found - proceed with analysis
    print(f"\n📁 Opening HAR temperature file...")
    print(f"   File: {har_file}")

    # Open the NetCDF file
    ds = xr.open_dataset(har_file)

    print("\n" + "="*80)
    print("📊 DATASET STRUCTURE")
    print("="*80)

    # Display basic info
    print(f"\nDataset dimensions:")
    for dim, size in ds.dims.items():
        print(f"   {dim}: {size}")

    print(f"\nDataset coordinates:")
    for coord in ds.coords:
        print(f"   {coord}: {ds.coords[coord].shape} - {ds.coords[coord].dtype}")
        if hasattr(ds.coords[coord], 'long_name'):
            print(f"      Long name: {ds.coords[coord].long_name}")
        if hasattr(ds.coords[coord], 'units'):
            print(f"      Units: {ds.coords[coord].units}")

    print(f"\nData variables:")
    for var in ds.data_vars:
        print(f"   {var}: {ds[var].shape} - {ds[var].dtype}")
        if hasattr(ds[var], 'long_name'):
            print(f"      Long name: {ds[var].long_name}")
        if hasattr(ds[var], 'units'):
            print(f"      Units: {ds[var].units}")
        if hasattr(ds[var], 'description'):
            print(f"      Description: {ds[var].description}")

    print(f"\nGlobal attributes:")
    for attr in ds.attrs:
        print(f"   {attr}: {ds.attrs[attr]}")

    print("\n" + "="*80)
    print("📊 DATA SAMPLE")
    print("="*80)

    # Show time range
    if 'time' in ds.coords:
        print(f"\nTime range:")
        print(f"   Start: {pd.Timestamp(ds.time.values[0])}")
        print(f"   End: {pd.Timestamp(ds.time.values[-1])}")
        print(f"   Total timesteps: {len(ds.time)}")
        print(f"   Temporal resolution: {ds.time.values[1] - ds.time.values[0]}")

    # Show spatial extent
    if 'lat' in ds.coords and 'lon' in ds.coords:
        print(f"\nSpatial extent:")
        print(f"   Latitude: {float(ds.lat.min()):.3f}° to {float(ds.lat.max()):.3f}°")
        print(f"   Longitude: {float(ds.lon.min()):.3f}° to {float(ds.lon.max()):.3f}°")
        print(f"   Grid cells: {len(ds.lat)} x {len(ds.lon)}")

    # Show sample temperature values (if t2 variable exists)
    if 't2' in ds.data_vars:
        print(f"\nTemperature statistics (first timestep):")
        t2_sample = ds['t2'].isel(time=0)
        print(f"   Min: {float(t2_sample.min()):.2f}")
        print(f"   Max: {float(t2_sample.max()):.2f}")
        print(f"   Mean: {float(t2_sample.mean()):.2f}")
        print(f"   Median: {float(t2_sample.median()):.2f}")

    print("\n" + "="*80)
    print("✅ Dataset structure check complete!")
    print("="*80)

    # Close the dataset
    ds.close()

🔍 Searching for HAR file in common locations...
✅ Found file at: /run/user/1001/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc

📁 Opening HAR temperature file...
   File: /run/user/1001/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc

📊 DATASET STRUCTURE

Dataset dimensions:
   time: 8784
   south_north: 252
   west_east: 381

Dataset coordinates:
   time: (8784,) - datetime64[ns]
      Long name: Time
   west_east: (381,) - float32
      Long name: x-coordinate in Cartesian system
      Units: m
   south_north: (252,) - float32
      Long name: y-coordinate in Cartesian system
      Units: m
   lon: (252, 381) - float32
      Long name: Longitude
      Units: degrees_east
   lat: (252, 381) - float32
      Long name: Latitude
      Units: degrees_north

Data variables:
   t2: (8784, 252, 381) - float32
      Long name: temp at 2 m
      Units: k

Global attributes:
   TITLE: HA

/tmp/ipykernel_26357/1820008716.py:81: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():
<frozen _collections_abc>:899: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.


   Min: 237.71
   Max: 295.19
   Mean: 263.98
   Median: 262.46

✅ Dataset structure check complete!


In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

# Define the directory and files
har_dir = Path("/home/jberg/OneDrive/Raven_worldwide/01_data/meteo/HAR")
files = [
    "HARv2_d10km_d_2d_potevap_1980.nc",
    "HARv2_d10km_d_2d_prcp_1980.nc",
    "HARv2_d10km_d_2d_t2_max_1980.nc",
    "HARv2_d10km_d_2d_t2_mean_1980.nc",
    "HARv2_d10km_d_2d_t2_min_1980.nc",
    "HARv2_d10km_static_hgt.nc"
]

print("🔍 Checking HAR NetCDF file structures...")
print("="*100)

for filename in files:
    filepath = har_dir / filename
    
    if not filepath.exists():
        print(f"\n❌ File not found: {filename}")
        continue
    
    print(f"\n{'='*100}")
    print(f"📁 FILE: {filename}")
    print(f"{'='*100}")
    
    try:
        # Open the dataset
        ds = xr.open_dataset(filepath)
        
        # File size
        file_size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f"\n💾 File size: {file_size_mb:.1f} MB")
        
        # Dimensions (use .sizes instead of .dims to avoid warning)
        print(f"\n📏 DIMENSIONS:")
        for dim, size in ds.sizes.items():
            print(f"   {dim:15s}: {size:6d}")
        
        # Coordinates
        print(f"\n🗺️  COORDINATES:")
        for coord in ds.coords:
            coord_data = ds.coords[coord]
            print(f"   {coord:15s}: shape={str(coord_data.shape):15s} dtype={coord_data.dtype}")
            
            # Show range for numeric coordinates
            if coord_data.dtype in [np.float32, np.float64, np.int32, np.int64]:
                if len(coord_data) > 0:
                    print(f"      → Range: {float(coord_data.min()):.4f} to {float(coord_data.max()):.4f}")
            
            # Show attributes
            if hasattr(coord_data, 'long_name'):
                print(f"      → Long name: {coord_data.long_name}")
            if hasattr(coord_data, 'units'):
                print(f"      → Units: {coord_data.units}")
        
        # Data Variables
        print(f"\n📊 DATA VARIABLES:")
        for var in ds.data_vars:
            var_data = ds[var]
            print(f"   {var:15s}: shape={str(var_data.shape):15s} dtype={var_data.dtype}")
            
            # Show attributes
            if hasattr(var_data, 'long_name'):
                print(f"      → Long name: {var_data.long_name}")
            if hasattr(var_data, 'units'):
                print(f"      → Units: {var_data.units}")
            if hasattr(var_data, 'description'):
                print(f"      → Description: {var_data.description}")
            
            # Show sample statistics
            try:
                sample = var_data.isel(time=0) if 'time' in var_data.dims else var_data
                valid_data = sample.values[~np.isnan(sample.values)]
                if len(valid_data) > 0:
                    print(f"      → Sample stats (first timestep):")
                    print(f"         Min: {valid_data.min():.4f}, Max: {valid_data.max():.4f}, Mean: {valid_data.mean():.4f}")
            except Exception as e:
                print(f"      → Could not compute stats: {e}")
        
        # Global Attributes
        print(f"\n🌍 GLOBAL ATTRIBUTES:")
        for attr in ds.attrs:
            attr_value = ds.attrs[attr]
            # Truncate long attributes
            if isinstance(attr_value, str) and len(attr_value) > 100:
                attr_value = attr_value[:100] + "..."
            print(f"   {attr:20s}: {attr_value}")
        
        # Time information
        if 'time' in ds.coords:
            print(f"\n⏰ TIME INFORMATION:")
            time_vals = ds.time.values
            print(f"   First timestep: {pd.Timestamp(time_vals[0])}")
            print(f"   Last timestep:  {pd.Timestamp(time_vals[-1])}")
            print(f"   Total steps:    {len(time_vals)}")
            if len(time_vals) > 1:
                dt = pd.Timestamp(time_vals[1]) - pd.Timestamp(time_vals[0])
                print(f"   Time step:      {dt}")
        
        # Spatial information
        if 'lat' in ds.coords and 'lon' in ds.coords:
            print(f"\n🗺️  SPATIAL INFORMATION:")
            print(f"   Latitude:  {float(ds.lat.min()):7.3f}° to {float(ds.lat.max()):7.3f}° ({len(ds.lat)} cells)")
            print(f"   Longitude: {float(ds.lon.min()):7.3f}° to {float(ds.lon.max()):7.3f}° ({len(ds.lon)} cells)")
            
            # Estimate resolution - FIX: properly extract scalar values
            if len(ds.lat) > 1 and len(ds.lon) > 1:
                lat_vals = ds.lat.values
                lon_vals = ds.lon.values
                lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                lon_res = abs(float(lon_vals[1] - lon_vals[0]))
                print(f"   Resolution: ~{lat_res:.4f}° x {lon_res:.4f}° (~{lat_res*111:.1f} km x {lon_res*111:.1f} km)")
        
        # Close dataset
        ds.close()
        
    except Exception as e:
        print(f"\n❌ Error reading file: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*100)
print("✅ All files checked!")
print("="*100)

🔍 Checking HAR NetCDF file structures...

📁 FILE: HARv2_d10km_d_2d_potevap_1980.nc

💾 File size: 106.4 MB

📏 DIMENSIONS:
   time           :    366
   south_north    :    252
   west_east      :    381

🗺️  COORDINATES:
   time           : shape=(366,)          dtype=datetime64[ns]
      → Long name: Time
   west_east      : shape=(381,)          dtype=float32
      → Range: -1675001.0000 to 2124999.0000
      → Long name: x-coordinate in Cartesian system
      → Units: m
   south_north    : shape=(252,)          dtype=float32
      → Range: -744999.0000 to 1765001.0000
      → Long name: y-coordinate in Cartesian system
      → Units: m
   lon            : shape=(252, 381)      dtype=float32
      → Range: 61.4748 to 110.0567
      → Long name: Longitude
      → Units: degrees_east
   lat            : shape=(252, 381)      dtype=float32
      → Range: 23.3861 to 47.7842
      → Long name: Latitude
      → Units: degrees_north

📊 DATA VARIABLES:
   potevap        : shape=(366, 252, 381

Traceback (most recent call last):
  File "/tmp/ipykernel_104607/2344054317.py", line 115, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_104607/2344054317.py", line 115, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_104607/2344054317.py", line 115, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_104607/2344054317.py", line 115, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^


💾 File size: 83.0 MB

📏 DIMENSIONS:
   time           :    367
   south_north    :    252
   west_east      :    381

🗺️  COORDINATES:
   west_east      : shape=(381,)          dtype=float32
      → Range: -1675001.0000 to 2124999.0000
      → Long name: x-coordinate in Cartesian system
      → Units: m
   south_north    : shape=(252,)          dtype=float32
      → Range: -744998.9375 to 1765001.0000
      → Long name: y-coordinate in Cartesian system
      → Units: m
   lon            : shape=(252, 381)      dtype=float32
      → Range: 61.4748 to 110.0567
      → Long name: Longitude
      → Units: degrees_east
   lat            : shape=(252, 381)      dtype=float32
      → Range: 23.3861 to 47.7842
      → Long name: Latitude
      → Units: degrees_north
   time           : shape=(367,)          dtype=datetime64[ns]
      → Long name: Time

📊 DATA VARIABLES:
   t2_min         : shape=(367, 252, 381) dtype=float32
      → Long name: Daily minimum 2-meter air temperature
      → Uni

Traceback (most recent call last):
  File "/tmp/ipykernel_104607/2344054317.py", line 115, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_104607/2344054317.py", line 115, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars


In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

# Path to your GloGEM NetCDF files
base_dir = Path('/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/snowline')
scenario = 'ssp126'
component = 'Discharge'  # Change to check others

# Load NetCDF
nc_file = base_dir / component / scenario / f'GloGEM_{component}_Indus_{scenario}.nc'
dat_dir = base_dir / component / scenario

print(f"Checking: {nc_file.name}")
print("=" * 70)

# Open NetCDF
ds = xr.open_dataset(nc_file)

# Get dimensions
n_glaciers = len(ds.glacier_id)
n_timesteps = len(ds.time)
var_name = component.lower()

print(f"\n📊 NetCDF Contents:")
print(f"   Number of glaciers: {n_glaciers}")
print(f"   Number of timesteps: {n_timesteps}")
print(f"   Date range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"   Data shape: {ds[var_name].shape}")
print(f"   File size: {nc_file.stat().st_size / 1024 / 1024:.2f} MB")

# Count non-NaN values
total_values = ds[var_name].size
non_nan_values = np.count_nonzero(~np.isnan(ds[var_name].values))
nan_percentage = (1 - non_nan_values / total_values) * 100

print(f"\n📈 Data Statistics:")
print(f"   Total cells: {total_values:,}")
print(f"   Non-NaN values: {non_nan_values:,}")
print(f"   NaN percentage: {nan_percentage:.1f}%")
print(f"   Data range: {np.nanmin(ds[var_name].values):.3f} to {np.nanmax(ds[var_name].values):.3f}")

# Check .dat files
dat_files = sorted(dat_dir.glob('*.dat'))
total_dat_size = sum(f.stat().st_size for f in dat_files) / 1024 / 1024

print(f"\n📁 Original .dat Files:")
print(f"   Number of files: {len(dat_files)}")
print(f"   Total size: {total_dat_size:.2f} MB")

# Estimate expected glaciers from .dat files
print(f"\n🔍 Checking first .dat file for comparison...")
first_dat = dat_files[0]
glacier_ids_in_dat = set()

with open(first_dat, 'r') as f:
    for line in f:
        if line.startswith("ID") or line.startswith("//") or line.strip() == "":
            continue
        parts = line.strip().split()
        if len(parts) >= 5:
            glacier_ids_in_dat.add(parts[0])

print(f"   File: {first_dat.name}")
print(f"   Glaciers in this file: {len(glacier_ids_in_dat)}")

# Sample some glacier IDs
sample_glacier_ids = list(ds.glacier_id.values[:5])
print(f"\n🆔 Sample glacier IDs from NetCDF:")
for gid in sample_glacier_ids:
    print(f"   {gid}")

# Check if sample glaciers have data
print(f"\n✅ Checking if sample glaciers have data:")
for gid in sample_glacier_ids[:3]:
    glacier_data = ds[var_name].sel(glacier_id=gid).values
    n_valid = np.count_nonzero(~np.isnan(glacier_data))
    print(f"   Glacier {gid}: {n_valid}/{len(glacier_data)} valid values ({n_valid/len(glacier_data)*100:.1f}%)")

# Close dataset
ds.close()

print("\n" + "=" * 70)
print("🤔 VERDICT:")
print(f"   Expected: 1000s of glaciers")
print(f"   Found: {n_glaciers} glaciers")
if n_glaciers < 100:
    print("   ❌ WARNING: Very few glaciers! Something went wrong.")
elif n_glaciers < 1000:
    print("   ⚠️  WARNING: Fewer glaciers than expected.")
else:
    print("   ✅ Glacier count looks reasonable")

if nan_percentage > 90:
    print(f"   ❌ WARNING: {nan_percentage:.1f}% of data is NaN!")
elif nan_percentage > 70:
    print(f"   ⚠️  High NaN percentage: {nan_percentage:.1f}%")
else:
    print(f"   ✅ NaN percentage looks OK: {nan_percentage:.1f}%")

compression_ratio = (1 - (nc_file.stat().st_size / 1024 / 1024) / total_dat_size) * 100
print(f"   Compression: {compression_ratio:.1f}% reduction")
if compression_ratio > 95:
    print("   ❌ WARNING: Compression ratio suspiciously high!")

Checking: GloGEM_Discharge_Indus_ssp126.nc

📊 NetCDF Contents:
   Number of glaciers: 19904
   Number of timesteps: 365
   Date range: 1979-10-01T00:00:00.000000000 to 1980-09-29T00:00:00.000000000
   Data shape: (365, 19904)
   File size: 6.00 MB

📈 Data Statistics:
   Total cells: 7,264,960
   Non-NaN values: 7,264,960
   NaN percentage: 0.0%
   Data range: 0.000 to 426.930

📁 Original .dat Files:
   Number of files: 95
   Total size: 4376.03 MB

🔍 Checking first .dat file for comparison...
   File: centralasiaW_Discharge__Indus_gl_all_03_merged.dat
   Glaciers in this file: 37

🆔 Sample glacier IDs from NetCDF:
   00001
   00002
   00003
   00004
   00005

✅ Checking if sample glaciers have data:
   Glacier 00001: 365/365 valid values (100.0%)
   Glacier 00002: 365/365 valid values (100.0%)
   Glacier 00003: 365/365 valid values (100.0%)

🤔 VERDICT:
   Expected: 1000s of glaciers
   Found: 19904 glaciers
   ✅ Glacier count looks reasonable
   ✅ NaN percentage looks OK: 0.0%
   Compr

In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path

# Path to your GloGEM NetCDF file
nc_path = Path('/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc')

# Open and inspect
ds = xr.open_dataset(nc_path)

print("=" * 60)
print("NETCDF FILE STRUCTURE")
print("=" * 60)

print("\n📊 Dimensions:")
for dim, size in ds.dims.items():
    print(f"  {dim}: {size}")

print("\n📋 Variables:")
for var in ds.data_vars:
    print(f"  {var}: {ds[var].dims} - {ds[var].shape}")
    print(f"    dtype: {ds[var].dtype}")
    if hasattr(ds[var], 'units'):
        print(f"    units: {ds[var].units}")

print("\n📍 Coordinates:")
for coord in ds.coords:
    print(f"  {coord}: {ds[coord].dims} - {ds[coord].shape}")
    print(f"    dtype: {ds[coord].dtype}")

print("\n🕐 Time information:")
time_values = pd.to_datetime(ds.time.values)
print(f"  First time: {time_values[0]}")
print(f"  Last time: {time_values[-1]}")
print(f"  Total timesteps: {len(time_values)}")
print(f"  Sample times (first 5): {time_values[:5].tolist()}")

print("\n🏔️ Glacier ID information:")
glacier_ids = ds.glacier_id.values.astype(str)
print(f"  Total glaciers: {len(glacier_ids)}")
print(f"  Sample glacier IDs (first 5): {glacier_ids[:5].tolist()}")
print(f"  Glacier ID type: {type(glacier_ids[0])}")

print("\n📦 Data variable (discharge):")
var_name = 'discharge'
if var_name in ds:
    print(f"  Shape: {ds[var_name].shape}")
    print(f"  Sample values (first glacier, first 5 times):")
    print(f"    {ds[var_name].values[:5, 0]}")
else:
    print(f"  Variable '{var_name}' not found!")
    print(f"  Available variables: {list(ds.data_vars)}")

print("\n🏷️ Attributes:")
for attr, value in ds.attrs.items():
    print(f"  {attr}: {value}")

ds.close()
print("\n✅ Done!")

NETCDF FILE STRUCTURE

📊 Dimensions:
  time: 44165
  glacier_id: 19904
  date: 44165

📋 Variables:
  discharge: ('time', 'glacier_id') - (44165, 19904)
    dtype: float32

📍 Coordinates:
  time: ('date',) - (44165,)
    dtype: datetime64[ns]
  glacier_id: ('glacier_id',) - (19904,)
    dtype: <U5

🕐 Time information:
  First time: 1979-10-01 00:00:00
  Last time: 2100-09-30 00:00:00
  Total timesteps: 44165
  Sample times (first 5): [Timestamp('1979-10-01 00:00:00'), Timestamp('1979-10-02 00:00:00'), Timestamp('1979-10-03 00:00:00'), Timestamp('1979-10-04 00:00:00'), Timestamp('1979-10-05 00:00:00')]

🏔️ Glacier ID information:
  Total glaciers: 19904
  Sample glacier IDs (first 5): ['00001', '00002', '00003', '00004', '00005']
  Glacier ID type: <class 'numpy.str_'>

📦 Data variable (discharge):
  Shape: (44165, 19904)
  Sample values (first glacier, first 5 times):


/tmp/ipykernel_297862/1746740166.py:16: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():
<frozen _collections_abc>:899: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.


    [0. 0. 0. 0. 0.]

🏷️ Attributes:
  title: GloGEM Discharge for Indus Basin
  scenario: ssp126
  source: GloGEM glacier model
  units: mm/day
  date_range: 1979-10-01T00:00:00.000000000 to 2100-09-30T00:00:00.000000000
  n_glaciers: 19904
  n_timesteps: 44165

✅ Done!


In [1]:
import xarray as xr
import numpy as np

# Open the NetCDF file
nc_path = '/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc'
ds = xr.open_dataset(nc_path)

print("="*60)
print("NETCDF FILE STRUCTURE")
print("="*60)

print("\n📊 DIMENSIONS:")
for dim_name, dim_size in ds.sizes.items():  # Use .sizes instead of .dims
    print(f"  {dim_name}: {dim_size}")

print("\n" + "="*60)
print("GLACIER IDs - SAMPLE")
print("="*60)

# Get glacier IDs
glacier_ids = ds.glacier_id.values

print(f"\n  Total number of glaciers: {len(glacier_ids)}")
print(f"  Glacier ID dtype: {glacier_ids.dtype}")

# Show first 10 glacier IDs
print(f"\n  First 10 glacier IDs:")
for i, gid in enumerate(glacier_ids[:10]):
    print(f"    [{i}] '{gid}'")

# Show last 10 glacier IDs
print(f"\n  Last 10 glacier IDs:")
for i in range(10):
    idx = len(glacier_ids) - 10 + i
    gid = glacier_ids[idx]
    print(f"    [{idx}] '{gid}'")

# Check for leading zeros (sample only)
print(f"\n  ID Format Analysis (first 100 IDs):")
glacier_ids_str = glacier_ids[:100].astype(str)
print(f"    Sample IDs: {list(glacier_ids_str[:10])}")

# Check if IDs have leading zeros
has_leading_zeros = any(id_str.startswith('0') and len(id_str) > 1 for id_str in glacier_ids_str)
print(f"    Contains leading zeros: {has_leading_zeros}")

# Check ID length distribution (sample)
sample_lengths = [len(str(gid)) for gid in glacier_ids[:1000]]
unique_lengths = sorted(set(sample_lengths))
print(f"    ID length distribution (first 1000):")
for length in unique_lengths:
    count = sample_lengths.count(length)
    print(f"      {length} digits: {count} glaciers")

print("\n" + "="*60)
print("TIME DIMENSION")
print("="*60)

time_vals = ds.time.values
print(f"  Total time steps: {len(time_vals)}")
print(f"  First date: {time_vals[0]}")
print(f"  Last date: {time_vals[-1]}")

print("\n" + "="*60)
print("DATA SAMPLE")
print("="*60)

# Get the discharge variable
discharge = ds['discharge']
print(f"\n  Discharge variable:")
print(f"    Shape: {discharge.shape}")
print(f"    Units: {discharge.attrs.get('units', 'N/A')}")

# Sample first glacier data
print(f"\n  Sample data (first glacier, first 5 time steps):")
sample_data = discharge.isel(glacier_id=0, time=slice(0, 5)).values
print(f"    {sample_data}")

ds.close()

print("\n" + "="*60)
print("✅ INSPECTION COMPLETE")
print("="*60)
print("\n🔍 KEY FINDING:")
print(f"  Glacier IDs are stored as STRINGS with format: '{glacier_ids[0]}'")
print(f"  Leading zeros: {'YES' if has_leading_zeros else 'NO'}")
print(f"  Total glaciers: {len(glacier_ids)}")

NETCDF FILE STRUCTURE

📊 DIMENSIONS:
  time: 44165
  glacier_id: 19904
  date: 44165

GLACIER IDs - SAMPLE

  Total number of glaciers: 19904
  Glacier ID dtype: <U5

  First 10 glacier IDs:
    [0] '00001'
    [1] '00002'
    [2] '00003'
    [3] '00004'
    [4] '00005'
    [5] '00006'
    [6] '00007'
    [7] '00008'
    [8] '00009'
    [9] '00010'

  Last 10 glacier IDs:
    [19894] '27976'
    [19895] '27977'
    [19896] '27978'
    [19897] '27979'
    [19898] '27980'
    [19899] '27981'
    [19900] '27982'
    [19901] '27983'
    [19902] '27984'
    [19903] '27988'

  ID Format Analysis (first 100 IDs):
    Sample IDs: [np.str_('00001'), np.str_('00002'), np.str_('00003'), np.str_('00004'), np.str_('00005'), np.str_('00006'), np.str_('00007'), np.str_('00008'), np.str_('00009'), np.str_('00010')]
    Contains leading zeros: True
    ID length distribution (first 1000):
      5 digits: 1000 glaciers

TIME DIMENSION
  Total time steps: 44165
  First date: 1979-10-01T00:00:00.000000000

In [9]:
import xarray as xr
import numpy as np

# RGI IDs to check
rgi_ids_to_check = [
    'RGI60-14.04481',
    'RGI60-14.02344',
    'RGI60-14.03405',
]

# Open GloGEM NetCDF
nc_path = '/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc'
ds = xr.open_dataset(nc_path)

# Get all glacier IDs from NetCDF (as strings with leading zeros)
nc_glacier_ids = ds.glacier_id.values.astype(str)

print("="*70)
print("CHECKING RGI IDs IN GLOGEM NETCDF")
print("="*70)

print(f"\n📊 NetCDF contains {len(nc_glacier_ids)} glacier IDs")
print(f"   Format: '{nc_glacier_ids[0]}' (type: {type(nc_glacier_ids[0])})")

print(f"\n🔍 Checking {len(rgi_ids_to_check)} RGI IDs...")
print("="*70)

# Extract numeric parts from RGI IDs (with leading zeros)
results = []
for rgi_id in rgi_ids_to_check:
    # Extract numeric part (e.g., "RGI60-14.03941" → "03941")
    numeric_part = rgi_id.split('.')[-1]
    
    # Check if it exists in NetCDF
    found = numeric_part in nc_glacier_ids
    
    results.append({
        'rgi_id': rgi_id,
        'numeric_id': numeric_part,
        'in_netcdf': found
    })
    
    # Print result
    status = "✅ FOUND" if found else "❌ NOT FOUND"
    print(f"{status:15s} {rgi_id:20s} → numeric: '{numeric_part}'")

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

found_count = sum(1 for r in results if r['in_netcdf'])
missing_count = len(results) - found_count

print(f"\n✅ Found in NetCDF: {found_count}/{len(results)}")
print(f"❌ Missing in NetCDF: {missing_count}/{len(results)}")

if missing_count > 0:
    print(f"\n🔍 Missing IDs:")
    for r in results:
        if not r['in_netcdf']:
            print(f"   - {r['rgi_id']} (numeric: '{r['numeric_id']}')")
    
    # Check if the numeric IDs exist WITHOUT leading zeros
    print(f"\n🔍 Checking if numeric IDs exist WITHOUT leading zeros...")
    for r in results:
        if not r['in_netcdf']:
            # Try without leading zeros
            numeric_no_zeros = str(int(r['numeric_id']))
            if numeric_no_zeros in nc_glacier_ids:
                print(f"   ⚠️  {r['rgi_id']}: '{r['numeric_id']}' not found, but '{numeric_no_zeros}' EXISTS!")
            else:
                print(f"   ❌ {r['rgi_id']}: Neither '{r['numeric_id']}' nor '{numeric_no_zeros}' found")

ds.close()

print("\n" + "="*70)
print("✅ Check complete!")
print("="*70)

CHECKING RGI IDs IN GLOGEM NETCDF

📊 NetCDF contains 19904 glacier IDs
   Format: '00001' (type: <class 'numpy.str_'>)

🔍 Checking 3 RGI IDs...
✅ FOUND         RGI60-14.04481       → numeric: '04481'
✅ FOUND         RGI60-14.02344       → numeric: '02344'
✅ FOUND         RGI60-14.03405       → numeric: '03405'

SUMMARY

✅ Found in NetCDF: 3/3
❌ Missing in NetCDF: 0/3

✅ Check complete!


In [2]:
import xarray as xr
import pandas as pd

# Open GloGEM NetCDF
nc_path = '/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc'
ds = xr.open_dataset(nc_path)

# Get glacier IDs
nc_glacier_ids = ds.glacier_id.values.astype(str)

print("="*70)
print("DEBUGGING GLACIER ID FORMAT IN NETCDF")
print("="*70)

# Check the specific IDs from the validation
test_ids = ['03941', '03544', '04233', '00766', '01451']

print(f"\n🔍 Checking format of glacier IDs in NetCDF:")
print(f"   Total glaciers in NetCDF: {len(nc_glacier_ids)}")
print(f"   First 20 glacier IDs:")
for i, gid in enumerate(nc_glacier_ids[:20]):
    print(f"      [{i:3d}] '{gid}' (type: {type(gid).__name__}, repr: {repr(gid)})")

print(f"\n🔍 Checking if test IDs exist (WITH leading zeros):")
for test_id in test_ids:
    exists = test_id in nc_glacier_ids
    print(f"   '{test_id}' → {'✅ FOUND' if exists else '❌ NOT FOUND'}")

print(f"\n🔍 Checking if test IDs exist (WITHOUT leading zeros):")
for test_id in test_ids:
    test_id_no_zeros = str(int(test_id))
    exists = test_id_no_zeros in nc_glacier_ids
    print(f"   '{test_id_no_zeros}' → {'✅ FOUND' if exists else '❌ NOT FOUND'}")

print(f"\n🔍 Searching for these specific glaciers in the NetCDF:")
for test_id in test_ids:
    # Try both formats
    found_with_zeros = test_id in nc_glacier_ids
    found_without_zeros = str(int(test_id)) in nc_glacier_ids
    
    if found_with_zeros:
        idx = list(nc_glacier_ids).index(test_id)
        print(f"   '{test_id}' → FOUND at index {idx}")
    elif found_without_zeros:
        idx = list(nc_glacier_ids).index(str(int(test_id)))
        print(f"   '{test_id}' → FOUND as '{str(int(test_id))}' at index {idx}")
    else:
        print(f"   '{test_id}' → NOT FOUND in any format")

# Check unique ID lengths
print(f"\n📏 Length distribution of glacier IDs:")
id_lengths = {}
for gid in nc_glacier_ids[:1000]:  # Sample first 1000
    length = len(gid)
    id_lengths[length] = id_lengths.get(length, 0) + 1

for length, count in sorted(id_lengths.items()):
    print(f"   {length} chars: {count} glaciers")

ds.close()

print("\n" + "="*70)
print("✅ Inspection complete!")
print("="*70)

DEBUGGING GLACIER ID FORMAT IN NETCDF

🔍 Checking format of glacier IDs in NetCDF:
   Total glaciers in NetCDF: 19904
   First 20 glacier IDs:
      [  0] '00001' (type: str_, repr: np.str_('00001'))
      [  1] '00002' (type: str_, repr: np.str_('00002'))
      [  2] '00003' (type: str_, repr: np.str_('00003'))
      [  3] '00004' (type: str_, repr: np.str_('00004'))
      [  4] '00005' (type: str_, repr: np.str_('00005'))
      [  5] '00006' (type: str_, repr: np.str_('00006'))
      [  6] '00007' (type: str_, repr: np.str_('00007'))
      [  7] '00008' (type: str_, repr: np.str_('00008'))
      [  8] '00009' (type: str_, repr: np.str_('00009'))
      [  9] '00010' (type: str_, repr: np.str_('00010'))
      [ 10] '00011' (type: str_, repr: np.str_('00011'))
      [ 11] '00012' (type: str_, repr: np.str_('00012'))
      [ 12] '00013' (type: str_, repr: np.str_('00013'))
      [ 13] '00014' (type: str_, repr: np.str_('00014'))
      [ 14] '00015' (type: str_, repr: np.str_('00015'))
  

In [7]:
import pandas as pd
from pathlib import Path

# Simulate what the validation function does
print("="*70)
print("SIMULATING VALIDATION LOGIC")
print("="*70)

# Load the glacier_id_mapping.csv (what validation uses)
mapping_path = Path('/home/jberg/OneDrive/Raven_worldwide/03_model_setups_coupled_test/catchment_0102/topo_files/glacier_id_mapping.csv')

if mapping_path.exists():
    mapping_df = pd.read_csv(mapping_path)
    
    print(f"\n📋 Glacier ID Mapping CSV:")
    print(f"   Rows: {len(mapping_df)}")
    print(f"   Columns: {list(mapping_df.columns)}")
    print(f"\n   First 10 rows:")
    print(mapping_df.head(10))
    
    # Check the specific glaciers
    test_rgi_ids = [
        'RGI60-14.04481',
        'RGI60-14.03544',
        'RGI60-14.04233',
        'RGI60-14.00766',
        'RGI60-14.01451'
    ]
    
    print(f"\n🔍 Checking test RGI IDs in mapping CSV:")
    for rgi_id in test_rgi_ids:
        match = mapping_df[mapping_df['RGIId'] == rgi_id]
        if len(match) > 0:
            numeric_id = str(match.iloc[0]['numeric_id']).zfill(5)
            print(f"   ✅ {rgi_id} → numeric_id: {numeric_id}")
        else:
            print(f"   ❌ {rgi_id} → NOT IN MAPPING CSV!")
    
    # Now simulate the validation comparison
    print(f"\n🔍 Simulating validation set operations:")
    
    # Create numeric_to_rgi mapping (what validation does)
    numeric_to_rgi = {}
    for _, row in mapping_df.iterrows():
        numeric_id = str(row['numeric_id']).zfill(5)
        rgi_id = row['RGIId']
        numeric_to_rgi[numeric_id] = rgi_id
    
    glacier_ids_needed = set(numeric_to_rgi.keys())
    
    # Load NetCDF glacier IDs
    import xarray as xr
    nc_path = '/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc'
    ds = xr.open_dataset(nc_path)
    glacier_ids_glogem = set(ds.glacier_id.values.astype(str))
    ds.close()
    
    print(f"   Glacier IDs needed (from mapping): {len(glacier_ids_needed)}")
    print(f"   Glacier IDs in GloGEM: {len(glacier_ids_glogem)}")
    
    # The set operations (exactly what validation does)
    missing_in_glogem = glacier_ids_needed - glacier_ids_glogem
    matched = glacier_ids_needed.intersection(glacier_ids_glogem)
    
    print(f"   Matched: {len(matched)}")
    print(f"   Missing in GloGEM: {len(missing_in_glogem)}")
    
    # Check our test IDs
    print(f"\n🔍 Checking test IDs after set operations:")
    for rgi_id in test_rgi_ids:
        # Find numeric ID from mapping
        match = mapping_df[mapping_df['RGIId'] == rgi_id]
        if len(match) > 0:
            numeric_id = str(match.iloc[0]['numeric_id']).zfill(5)
            
            in_needed = numeric_id in glacier_ids_needed
            in_glogem = numeric_id in glacier_ids_glogem
            in_matched = numeric_id in matched
            in_missing = numeric_id in missing_in_glogem
            
            print(f"   {rgi_id} ({numeric_id}):")
            print(f"      In needed set: {in_needed}")
            print(f"      In GloGEM set: {in_glogem}")
            print(f"      In matched: {in_matched}")
            print(f"      In missing: {in_missing}")
    
    # Show sample of missing glaciers
    print(f"\n📋 Sample of 'missing' glaciers (first 20):")
    for i, numeric_id in enumerate(list(missing_in_glogem)[:20]):
        rgi_id = numeric_to_rgi.get(numeric_id, 'UNKNOWN')
        print(f"      {numeric_id} → {rgi_id}")
    
else:
    print(f"❌ Mapping CSV not found: {mapping_path}")

print("\n" + "="*70)
print("✅ Simulation complete!")
print("="*70)

SIMULATING VALIDATION LOGIC

📋 Glacier ID Mapping CSV:
   Rows: 2328
   Columns: ['numeric_id', 'RGIId', 'area_km2', 'is_large']

   First 10 rows:
   numeric_id           RGIId  area_km2  is_large
0           1  RGI60-14.04581     0.034     False
1           2  RGI60-14.04583     0.050     False
2           3  RGI60-14.04582     0.077     False
3           4  RGI60-14.04586     0.349     False
4           5  RGI60-14.10840     0.246     False
5           6  RGI60-14.04577     0.068     False
6           7  RGI60-14.00029     1.277     False
7           8  RGI60-14.04571     0.085     False
8           9  RGI60-14.04567     0.115     False
9          10  RGI60-14.04555     0.039     False

🔍 Checking test RGI IDs in mapping CSV:
   ✅ RGI60-14.04481 → numeric_id: 00447
   ✅ RGI60-14.03544 → numeric_id: 00275
   ✅ RGI60-14.04233 → numeric_id: 00502
   ✅ RGI60-14.00766 → numeric_id: 02058
   ✅ RGI60-14.01451 → numeric_id: 01839

🔍 Simulating validation set operations:
   Glacier IDs neede

In [10]:
import pandas as pd
import xarray as xr
from pathlib import Path
import numpy as np

print("="*70)
print("CHECKING ALL GLACIERS FROM MAPPING CSV AGAINST GLOGEM NETCDF")
print("="*70)

# Load glacier_id_mapping.csv
mapping_path = Path('/home/jberg/OneDrive/Raven_worldwide//03_model_setups_coupled_test/catchment_0102/topo_files/glacier_id_mapping.csv')
mapping_df = pd.read_csv(mapping_path)

print(f"\n📋 Loaded {len(mapping_df)} glaciers from mapping CSV")

# Create numeric_to_rgi mapping
numeric_to_rgi = {}
for _, row in mapping_df.iterrows():
    numeric_id = str(row['numeric_id']).zfill(5)  # Pad to 5 digits
    rgi_id = row['RGIId']
    numeric_to_rgi[numeric_id] = rgi_id

glacier_ids_needed = set(numeric_to_rgi.keys())

print(f"   Glacier IDs needed (padded to 5 digits): {len(glacier_ids_needed)}")
print(f"   Sample needed IDs: {list(glacier_ids_needed)[:10]}")
print(f"   Type of needed IDs: {type(list(glacier_ids_needed)[0])}")

# Load GloGEM NetCDF
nc_path = '/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc'
ds = xr.open_dataset(nc_path)

# ❌ OLD WAY (BROKEN):
glacier_ids_glogem_BROKEN = set(ds.glacier_id.values.astype(str))

# ✅ NEW WAY (FIXED):
glacier_ids_glogem = set(str(x) for x in ds.glacier_id.values.astype(str))

ds.close()

print(f"\n📊 GloGEM NetCDF contains {len(glacier_ids_glogem)} glaciers")
print(f"   Sample GloGEM IDs (BROKEN): {list(glacier_ids_glogem_BROKEN)[:10]}")
print(f"   Type (BROKEN): {type(list(glacier_ids_glogem_BROKEN)[0])}")
print(f"   Sample GloGEM IDs (FIXED): {list(glacier_ids_glogem)[:10]}")
print(f"   Type (FIXED): {type(list(glacier_ids_glogem)[0])}")

# Compare sets - BROKEN WAY
missing_BROKEN = glacier_ids_needed - glacier_ids_glogem_BROKEN
matched_BROKEN = glacier_ids_needed.intersection(glacier_ids_glogem_BROKEN)

# Compare sets - FIXED WAY
missing_in_glogem = glacier_ids_needed - glacier_ids_glogem
matched = glacier_ids_needed.intersection(glacier_ids_glogem)

print(f"\n" + "="*70)
print("RESULTS - BROKEN vs FIXED")
print("="*70)
print(f"❌ BROKEN (numpy strings):")
print(f"   Matched: {len(matched_BROKEN)}/{len(glacier_ids_needed)} ({len(matched_BROKEN)/len(glacier_ids_needed)*100:.1f}%)")
print(f"   Missing: {len(missing_BROKEN)}/{len(glacier_ids_needed)} ({len(missing_BROKEN)/len(glacier_ids_needed)*100:.1f}%)")

print(f"\n✅ FIXED (Python strings):")
print(f"   Matched: {len(matched)}/{len(glacier_ids_needed)} ({len(matched)/len(glacier_ids_needed)*100:.1f}%)")
print(f"   Missing: {len(missing_in_glogem)}/{len(glacier_ids_needed)} ({len(missing_in_glogem)/len(glacier_ids_needed)*100:.1f}%)")

if len(missing_in_glogem) > 0:
    print(f"\n❌ FIRST 20 ACTUALLY MISSING GLACIERS:")
    for i, numeric_id in enumerate(list(missing_in_glogem)[:20]):
        rgi_id = numeric_to_rgi[numeric_id]
        print(f"   {i+1:3d}. '{numeric_id}' → {rgi_id}")

print(f"\n" + "="*70)
print("✅ THE BUG WAS: numpy.str_ vs str TYPE MISMATCH!")
print("="*70)

CHECKING ALL GLACIERS FROM MAPPING CSV AGAINST GLOGEM NETCDF

📋 Loaded 2328 glaciers from mapping CSV
   Glacier IDs needed (padded to 5 digits): 2328
   Sample needed IDs: ['01890', '00674', '00702', '00485', '01019', '01490', '01726', '01374', '00301', '00525']
   Type of needed IDs: <class 'str'>

📊 GloGEM NetCDF contains 19904 glaciers
   Sample GloGEM IDs (BROKEN): [np.str_('21205'), np.str_('00485'), np.str_('21392'), np.str_('07501'), np.str_('17972'), np.str_('23402'), np.str_('12081'), np.str_('26938'), np.str_('01816'), np.str_('08017')]
   Type (BROKEN): <class 'numpy.str_'>
   Sample GloGEM IDs (FIXED): ['21205', '00485', '21392', '07501', '17972', '23402', '12081', '26938', '01816', '08017']
   Type (FIXED): <class 'str'>

RESULTS - BROKEN vs FIXED
❌ BROKEN (numpy strings):
   Matched: 1442/2328 (61.9%)
   Missing: 886/2328 (38.1%)

✅ FIXED (Python strings):
   Matched: 1442/2328 (61.9%)
   Missing: 886/2328 (38.1%)

❌ FIRST 20 ACTUALLY MISSING GLACIERS:
     1. '01561' → 

In [11]:
import pandas as pd
import xarray as xr
from pathlib import Path

print("="*70)
print("CHECKING IF 'MISSING' GLACIERS ARE ACTUALLY IN GLOGEM")
print("="*70)

# Load mapping
mapping_path = Path('/home/jberg/OneDrive/Raven_worldwide//03_model_setups_coupled_test/catchment_0102/topo_files/glacier_id_mapping.csv')
mapping_df = pd.read_csv(mapping_path)

numeric_to_rgi = {}
for _, row in mapping_df.iterrows():
    numeric_id = str(row['numeric_id']).zfill(5)
    rgi_id = row['RGIId']
    numeric_to_rgi[numeric_id] = rgi_id

glacier_ids_needed = set(numeric_to_rgi.keys())

# Load GloGEM
nc_path = '/home/jberg/OneDrive/Raven_worldwide/01_data/GloGEM/Indus/GMB/GloGEM_Discharge_Indus_ssp126.nc'
ds = xr.open_dataset(nc_path)
glacier_ids_glogem = set(str(x) for x in ds.glacier_id.values)
ds.close()

# Find missing
missing_in_glogem = glacier_ids_needed - glacier_ids_glogem

print(f"\n📋 According to set operation: {len(missing_in_glogem)} missing")

# NOW CHECK MANUALLY IF THEY'RE ACTUALLY MISSING
print(f"\n🔍 Manually checking first 20 'missing' glaciers...")
print(f"="*70)

nc_ids_list = list(glacier_ids_glogem)  # Convert to list for searching

for i, numeric_id in enumerate(list(missing_in_glogem)[:20]):
    rgi_id = numeric_to_rgi[numeric_id]
    
    # Check multiple ways
    in_set = numeric_id in glacier_ids_glogem
    in_list = numeric_id in nc_ids_list
    
    # Try direct search
    ds = xr.open_dataset(nc_path)
    try:
        # Try to select this glacier
        test = ds.sel(glacier_id=numeric_id)
        can_select = True
    except:
        can_select = False
    ds.close()
    
    status = "✅ FOUND" if (in_set or in_list or can_select) else "❌ TRULY MISSING"
    print(f"{i+1:3d}. {rgi_id} ({numeric_id}): {status}")
    print(f"      in_set={in_set}, in_list={in_list}, can_select={can_select}")

print(f"\n" + "="*70)
print("✅ Manual check complete!")
print("="*70)

CHECKING IF 'MISSING' GLACIERS ARE ACTUALLY IN GLOGEM

📋 According to set operation: 886 missing

🔍 Manually checking first 20 'missing' glaciers...
  1. RGI60-14.02344 (01561): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  2. RGI60-14.03405 (00674): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  3. RGI60-14.05451 (00263): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  4. RGI60-14.11221 (00154): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  5. RGI60-14.04481 (00447): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  6. RGI60-14.11219 (00158): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  7. RGI60-14.04590 (00385): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  8. RGI60-14.05235 (00404): ❌ TRULY MISSING
      in_set=False, in_list=False, can_select=False
  9. RGI60-14.04575 (00147): ❌ TRULY MISSING
      in_set=False, in_list=Fa

In [1]:
import pandas as pd
from pathlib import Path

print("="*70)
print("🔍 THE PROBLEM: MISMATCHED NUMERIC IDs IN MAPPING CSV!")
print("="*70)

# Load the mapping CSV
mapping_path = Path('/home/jberg/OneDrive/Raven_worldwide/03_model_setups_coupled_test/catchment_0102/topo_files/glacier_id_mapping.csv')
mapping_df = pd.read_csv(mapping_path)

print(f"\n📋 Mapping CSV has {len(mapping_df)} glaciers")
print(f"   Columns: {list(mapping_df.columns)}")

# Check the test RGI IDs
test_rgi_ids = ['RGI60-14.04481', 'RGI60-14.02344', 'RGI60-14.03405']

print(f"\n🔍 Checking test RGI IDs in the mapping CSV:")
print(f"="*70)

for rgi_id in test_rgi_ids:
    match = mapping_df[mapping_df['RGIId'] == rgi_id]
    
    if len(match) > 0:
        row = match.iloc[0]
        numeric_id_in_csv = str(row['numeric_id']).zfill(5)
        
        # Extract the REAL numeric part from RGI ID
        real_numeric = rgi_id.split('.')[-1]
        
        matches = "✅ MATCH" if numeric_id_in_csv == real_numeric else "❌ MISMATCH"
        
        print(f"\n{rgi_id}:")
        print(f"   CSV numeric_id:     '{numeric_id_in_csv}'")
        print(f"   RGI ID numeric:     '{real_numeric}'")
        print(f"   Status:             {matches}")
        
        if matches == "❌ MISMATCH":
            print(f"   ⚠️  THE NUMERIC_ID IN THE CSV IS WRONG!")
    else:
        print(f"\n{rgi_id}: ❌ NOT IN CSV!")

print(f"\n" + "="*70)
print("💡 SOLUTION:")
print("="*70)
print("The numeric_id column in glacier_id_mapping.csv should contain")
print("the numeric part of the RGI ID (e.g., '04481' for RGI60-14.04481),")
print("NOT a sequential counter!")
print()
print("You need to regenerate the glacier_id_mapping.csv with:")
print("  numeric_id = RGIId.split('.')[-1]")
print("="*70)

🔍 THE PROBLEM: MISMATCHED NUMERIC IDs IN MAPPING CSV!

📋 Mapping CSV has 2328 glaciers
   Columns: ['numeric_id', 'RGIId', 'area_km2', 'is_large']

🔍 Checking test RGI IDs in the mapping CSV:

RGI60-14.04481:
   CSV numeric_id:     '00447'
   RGI ID numeric:     '04481'
   Status:             ❌ MISMATCH
   ⚠️  THE NUMERIC_ID IN THE CSV IS WRONG!

RGI60-14.02344:
   CSV numeric_id:     '01561'
   RGI ID numeric:     '02344'
   Status:             ❌ MISMATCH
   ⚠️  THE NUMERIC_ID IN THE CSV IS WRONG!

RGI60-14.03405:
   CSV numeric_id:     '00674'
   RGI ID numeric:     '03405'
   Status:             ❌ MISMATCH
   ⚠️  THE NUMERIC_ID IN THE CSV IS WRONG!

💡 SOLUTION:
The numeric_id column in glacier_id_mapping.csv should contain
the numeric part of the RGI ID (e.g., '04481' for RGI60-14.04481),
NOT a sequential counter!

You need to regenerate the glacier_id_mapping.csv with:
  numeric_id = RGIId.split('.')[-1]
